# Layer 2 starter notebook

Notebook này giúp bạn bắt đầu Layer 2 theo đúng export contract đã patch:

- đọc `layer2_residual_design`
- đọc `layer2_feature_manifest`
- dùng đúng feature whitelist
- train riêng cho `within_quarter_origin` = 1, 2, 3
- backtest theo expanding window
- so sánh Hybrid với DFM-only

**Lưu ý quan trọng về thời gian**

- `vintage_period` là **tháng** với semantics **first day of month**
- `target_quarter` là **quý** với semantics **first day of quarter**
- không convert sang month-end


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

LAYER1_OUTPUT_DIR = Path('outputs/layer1_dfm')
LAYER2_OUTPUT_DIR = Path('outputs/layer2')
PRIMARY_TARGET = 'dfm_residual_third_release'
BASE_NOWCAST_COL = 'dfm_nowcast'
ORIGIN_COL = 'within_quarter_origin'
TIME_ORDER_COL = 'vintage_period'
MIN_TRAIN_SIZE = 40
TEST_BLOCK_SIZE = 8


In [ ]:
design_parquet = LAYER1_OUTPUT_DIR / 'layer2_residual_design.parquet'
design_csv = LAYER1_OUTPUT_DIR / 'layer2_residual_design.csv'
manifest_csv = LAYER1_OUTPUT_DIR / 'layer2_feature_manifest.csv'
contract_json = LAYER1_OUTPUT_DIR / 'layer2_data_contract.json'

assert manifest_csv.exists(), 'Missing layer2_feature_manifest.csv'
assert contract_json.exists(), 'Missing layer2_data_contract.json'
assert design_parquet.exists() or design_csv.exists(), 'Missing layer2_residual_design.parquet/csv'

if design_parquet.exists():
    design = pd.read_parquet(design_parquet)
else:
    design = pd.read_csv(design_csv)

manifest = pd.read_csv(manifest_csv)
with open(contract_json, 'r', encoding='utf-8') as f:
    contract = json.load(f)

if 'vintage_period' in design.columns:
    design['vintage_period'] = design['vintage_period'].astype(str)
if 'target_quarter' in design.columns:
    design['target_quarter'] = design['target_quarter'].astype(str)

print('design shape:', design.shape)
print('manifest shape:', manifest.shape)
print('primary target from contract:', contract.get('primary_target'))
display(design.head())
display(manifest.head(20))
contract


## Feature whitelist

Chỉ dùng các cột được manifest cho phép. Không tự tiện thêm truth columns, metadata columns, hoặc audit-only fields vào X.

In [ ]:
feature_cols = manifest.loc[
    (manifest['role'] == 'feature') & (manifest['included_in_training_matrix'] == True),
    'column'
].astype(str).tolist()

primary_target = contract.get('primary_target', PRIMARY_TARGET)
forbidden_cols = contract.get('forbidden_feature_columns', [])

print('n features:', len(feature_cols))
print('first 20 features:', feature_cols[:20])
print('n forbidden columns:', len(forbidden_cols))

train_df = design.loc[design[primary_target].notna()].copy()
train_df = train_df.sort_values([TIME_ORDER_COL, 'target_quarter']).reset_index(drop=True)

print('trainable rows:', len(train_df))
train_df[[TIME_ORDER_COL, 'target_quarter', ORIGIN_COL, primary_target]].head()


## Mapping từ paper sang file thực tế

- `\hat g^{DFM}_{v,q}` -> `dfm_nowcast`
- `F_{m_v}` -> `state__current_smoothed__*`
- `F_{m_{v-1}}` -> `state__previous_vintage_smoothed__*`
- `N_{1:B,v,q}` -> `news_signed__*`
- `|N_{1:B,v,q}|` -> `news_abs__*`
- `c_{1:B,v,q}` -> `coverage__*`

Trong patch hiện tại, same-τ residual lags **không được export sẵn** để tránh leakage vô ý. Bạn có thể thêm sau, nhưng phải sinh chúng **bên trong backtest loop** từ quá khứ đã biết truth.

In [ ]:
def rmsfe(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def expanding_splits(n_obs, min_train_size=MIN_TRAIN_SIZE, test_block_size=TEST_BLOCK_SIZE):
    start = min_train_size
    while start < n_obs:
        train_end = start
        test_end = min(start + test_block_size, n_obs)
        train_idx = np.arange(0, train_end)
        test_idx = np.arange(train_end, test_end)
        if len(test_idx) == 0:
            break
        yield train_idx, test_idx
        start = test_end


def make_models():
    return {
        'elastic_net': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000, random_state=42)),
        ]),
        'gbr': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', GradientBoostingRegressor(n_estimators=300, learning_rate=0.03, max_depth=2, random_state=42)),
        ]),
        'rf': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(n_estimators=500, max_depth=4, min_samples_leaf=5, random_state=42, n_jobs=-1)),
        ]),
    }


def backtest_one_tau(df_tau, feature_cols, target_col, model_name, model):
    df_tau = df_tau.sort_values([TIME_ORDER_COL, 'target_quarter']).reset_index(drop=True)
    X = df_tau[feature_cols].copy()
    y = df_tau[target_col].astype(float).to_numpy()

    pred_frames = []
    for train_idx, test_idx in expanding_splits(len(df_tau)):
        X_train = X.iloc[train_idx]
        y_train = y[train_idx]
        X_test = X.iloc[test_idx]

        fitted = clone(model)
        fitted.fit(X_train, y_train)
        y_hat = fitted.predict(X_test)

        fold = df_tau.iloc[test_idx][['vintage_period', 'target_quarter', 'within_quarter_origin', BASE_NOWCAST_COL, target_col]].copy()
        fold['residual_hat'] = y_hat
        fold['hybrid_nowcast_hat'] = fold[BASE_NOWCAST_COL] + fold['residual_hat']
        pred_frames.append(fold)

    pred_df = pd.concat(pred_frames, ignore_index=True)

    residual_true = pred_df[target_col].to_numpy(dtype=float)
    residual_pred = pred_df['residual_hat'].to_numpy(dtype=float)
    hybrid_error = residual_true - residual_pred
    dfm_only_error = residual_true

    metrics = {
        'n_oos': int(len(pred_df)),
        'residual_rmsfe': rmsfe(residual_true, residual_pred),
        'residual_mae': float(mean_absolute_error(residual_true, residual_pred)),
        'hybrid_rmsfe': rmsfe(np.zeros_like(hybrid_error), hybrid_error),
        'hybrid_mae': float(mean_absolute_error(np.zeros_like(hybrid_error), hybrid_error)),
        'dfm_only_rmsfe': rmsfe(np.zeros_like(dfm_only_error), dfm_only_error),
        'dfm_only_mae': float(mean_absolute_error(np.zeros_like(dfm_only_error), dfm_only_error)),
    }
    metrics['rmsfe_gain_vs_dfm_pct'] = 100.0 * (metrics['dfm_only_rmsfe'] - metrics['hybrid_rmsfe']) / metrics['dfm_only_rmsfe']
    return pred_df, metrics


## Backtest riêng cho từng `within_quarter_origin`

In [ ]:
models = make_models()
all_pred = []
metric_rows = []

for tau in sorted(train_df[ORIGIN_COL].dropna().astype(int).unique().tolist()):
    df_tau = train_df.loc[train_df[ORIGIN_COL].astype(int) == int(tau)].copy()
    print(f'\n=== tau={tau} | rows={len(df_tau)} ===')

    for model_name, model in models.items():
        pred_df, metrics = backtest_one_tau(df_tau, feature_cols, primary_target, model_name, model)
        pred_df['model_name'] = model_name
        pred_df['tau'] = tau
        all_pred.append(pred_df)

        row = {'tau': tau, 'model_name': model_name, **metrics}
        metric_rows.append(row)
        print(row)

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.concat(all_pred, ignore_index=True)

display(metrics_df.sort_values(['tau', 'hybrid_rmsfe']))
display(predictions_df.head())


In [ ]:
LAYER2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(LAYER2_OUTPUT_DIR / 'layer2_metrics_summary.csv', index=False)
predictions_df.to_csv(LAYER2_OUTPUT_DIR / 'layer2_predictions_oos.csv', index=False)

run_config = {
    'primary_target': primary_target,
    'n_features': len(feature_cols),
    'feature_columns': feature_cols,
    'models': list(models.keys()),
    'min_train_size': MIN_TRAIN_SIZE,
    'test_block_size': TEST_BLOCK_SIZE,
}
with open(LAYER2_OUTPUT_DIR / 'layer2_run_config.json', 'w', encoding='utf-8') as f:
    json.dump(run_config, f, indent=2)

print('Saved to outputs/layer2/')


## Việc tiếp theo sau notebook này

1. Chạy lại với tuning tốt hơn bằng nested rolling CV.
2. So sánh `Hybrid` với `DFM-only` theo từng tau.
3. Chạy robustness target: `dfm_residual_latest_rtdsm`, `dfm_residual_gdpplus`.
4. Thêm interpretability: SHAP hoặc permutation importance.
5. Nếu muốn đúng hơn với notation trong paper, bạn có thể thêm same-τ residual lags **bên trong loop backtest**, không export sẵn toàn bảng.
